# Round 5 — Data Exploration

Loads raw `prices_round_5_day_*.csv` and `trades_round_5_day_*.csv` from `backtester/datasets/round5/`.

Round 5 introduces **10 product families × 5 variants each** = 50 instruments, all priced in `XIRECS`. No underlying/option structure — instead, variants within a family likely share a common factor (family mid).

Families:
- **GALAXY_SOUNDS** — BLACK_HOLES, DARK_MATTER, PLANETARY_RINGS, SOLAR_FLAMES, SOLAR_WINDS
- **MICROCHIP** — CIRCLE, OVAL, RECTANGLE, SQUARE, TRIANGLE
- **OXYGEN_SHAKE** — CHOCOLATE, EVENING_BREATH, GARLIC, MINT, MORNING_BREATH
- **PANEL** — 1X2, 1X4, 2X2, 2X4, 4X4
- **PEBBLES** — XS, S, M, L, XL
- **ROBOT** — DISHES, IRONING, LAUNDRY, MOPPING, VACUUMING
- **SLEEP_POD** — COTTON, LAMB_WOOL, NYLON, POLYESTER, SUEDE
- **SNACKPACK** — CHOCOLATE, PISTACHIO, RASPBERRY, STRAWBERRY, VANILLA
- **TRANSLATOR** — ASTRO_BLACK, ECLIPSE_CHARCOAL, GRAPHITE_MIST, SPACE_GRAY, VOID_BLUE
- **UV_VISOR** — AMBER, MAGENTA, ORANGE, RED, YELLOW

In [ ]:
# --- Config ---
DATASET_DIR = "../backtester/datasets/round5/"
DAYS = [2, 3, 4]

FAMILIES = {
    "GALAXY_SOUNDS": ["BLACK_HOLES", "DARK_MATTER", "PLANETARY_RINGS", "SOLAR_FLAMES", "SOLAR_WINDS"],
    "MICROCHIP":     ["CIRCLE", "OVAL", "RECTANGLE", "SQUARE", "TRIANGLE"],
    "OXYGEN_SHAKE":  ["CHOCOLATE", "EVENING_BREATH", "GARLIC", "MINT", "MORNING_BREATH"],
    "PANEL":         ["1X2", "1X4", "2X2", "2X4", "4X4"],
    "PEBBLES":       ["XS", "S", "M", "L", "XL"],
    "ROBOT":         ["DISHES", "IRONING", "LAUNDRY", "MOPPING", "VACUUMING"],
    "SLEEP_POD":     ["COTTON", "LAMB_WOOL", "NYLON", "POLYESTER", "SUEDE"],
    "SNACKPACK":     ["CHOCOLATE", "PISTACHIO", "RASPBERRY", "STRAWBERRY", "VANILLA"],
    "TRANSLATOR":    ["ASTRO_BLACK", "ECLIPSE_CHARCOAL", "GRAPHITE_MIST", "SPACE_GRAY", "VOID_BLUE"],
    "UV_VISOR":      ["AMBER", "MAGENTA", "ORANGE", "RED", "YELLOW"],
}
ALL_PRODUCTS = [f"{fam}_{v}" for fam, vs in FAMILIES.items() for v in vs]
print(f"{len(FAMILIES)} families, {len(ALL_PRODUCTS)} products total")

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.family": "monospace",
})

print("Ready.")

## 1. Load Prices (order book snapshots)

In [ ]:
def load_prices(dataset_dir, days):
    frames = []
    for d in days:
        path = Path(dataset_dir) / f"prices_round_5_day_{d}.csv"
        if not path.exists():
            print(f"Missing: {path}")
            continue
        df = pd.read_csv(path, sep=";")
        df["_day"] = d
        df["_global_ts"] = d * 1_000_000 + df["timestamp"].astype(np.int64)
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

prices = load_prices(DATASET_DIR, DAYS)
print(f"Loaded {len(prices):,} price rows")
if len(prices):
    products = sorted(prices["product"].unique())
    print(f"Products ({len(products)}):")
    for p in products:
        print(f"  {p}")

# Pivot to ts × product → mid_price (used by every downstream cell, O(1) lookups)
mid_wide = (
    prices.pivot_table(index="_global_ts", columns="product", values="mid_price", aggfunc="first")
    if len(prices) else pd.DataFrame()
)
print(f"mid_wide shape: {mid_wide.shape}")

## 2. Per-product price summary

In [ ]:
if len(mid_wide):
    summary = mid_wide.agg(["count", "mean", "min", "max", "std"]).T
    summary = summary.rename(columns={"count": "n", "std": "stdev"})
    summary["n"] = summary["n"].astype(int)
    for fam, variants in FAMILIES.items():
        cols = [f"{fam}_{v}" for v in variants if f"{fam}_{v}" in summary.index]
        print(f"\n--- {fam} ---")
        print(summary.loc[cols].round(2).to_string())

## 3. Mid price per family — variants overlaid

One subplot per family, all 5 variants on the same axis. Tight co-movement → suggests a shared factor / family mid that variants oscillate around.

In [ ]:
if len(mid_wide):
    fig, axes = plt.subplots(5, 2, figsize=(16, 22), sharex=True)
    cmap = plt.get_cmap("tab10")
    ts = mid_wide.index.to_numpy()
    for ax, (fam, variants) in zip(axes.flat, FAMILIES.items()):
        for i, v in enumerate(variants):
            col = f"{fam}_{v}"
            if col not in mid_wide.columns:
                continue
            ax.plot(ts, mid_wide[col].to_numpy(), linewidth=0.6, color=cmap(i), label=v)
        for d in DAYS[1:]:
            ax.axvline(d * 1_000_000, color="gray", linestyle=":", linewidth=0.5)
        ax.set_title(fam)
        ax.legend(fontsize=7, ncol=5, loc="upper right")
        ax.grid(True, alpha=0.25)
    fig.suptitle("Mid price per family", y=1.00)
    plt.tight_layout()
    plt.show()

## 4. Within-family spread (variant − family mean)

If variants share a common factor, the deviation (variant_mid − mean of family at that ts) is the **idiosyncratic component**. Persistent deviations suggest mean-reverting pair/basket trades within the family.

In [ ]:
if len(mid_wide):
    fig, axes = plt.subplots(5, 2, figsize=(16, 22), sharex=True)
    cmap = plt.get_cmap("tab10")
    ts = mid_wide.index.to_numpy()
    for ax, (fam, variants) in zip(axes.flat, FAMILIES.items()):
        cols = [f"{fam}_{v}" for v in variants if f"{fam}_{v}" in mid_wide.columns]
        if not cols:
            continue
        sub = mid_wide[cols]
        fam_mean = sub.mean(axis=1)
        for i, v in enumerate(variants):
            col = f"{fam}_{v}"
            if col not in sub.columns:
                continue
            ax.plot(ts, (sub[col] - fam_mean).to_numpy(),
                    linewidth=0.5, color=cmap(i), label=v, alpha=0.85)
        ax.axhline(0, color="black", linewidth=0.4)
        for d in DAYS[1:]:
            ax.axvline(d * 1_000_000, color="gray", linestyle=":", linewidth=0.5)
        ax.set_title(f"{fam} — variant minus family mean")
        ax.legend(fontsize=7, ncol=5, loc="upper right")
        ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

## 5. Within-family correlation matrix

Pearson correlation of 1-tick mid-price changes across the 5 variants. High positive corr → shared factor. Negative or near-zero corr → independent products that happen to share a name.

In [ ]:
if len(mid_wide):
    diffs_wide = mid_wide.diff()
    for fam, variants in FAMILIES.items():
        cols = [f"{fam}_{v}" for v in variants if f"{fam}_{v}" in diffs_wide.columns]
        if len(cols) < 2:
            continue
        cm = diffs_wide[cols].corr()
        print(f"\n{fam}")
        print(cm.round(3).to_string())

## 6. Bid/Ask spread per product

In [ ]:
if len(prices):
    sp = prices.dropna(subset=["bid_price_1", "ask_price_1"]).copy()
    sp["spread"] = sp["ask_price_1"] - sp["bid_price_1"]
    out = sp.groupby("product")["spread"].agg(["count", "mean", "min", "max"]).sort_values("mean", ascending=False)
    out["count"] = out["count"].astype(int)
    print(out.round(2).to_string())

## 7. Load Trades

In [ ]:
def load_trades(dataset_dir, days):
    frames = []
    for d in days:
        path = Path(dataset_dir) / f"trades_round_5_day_{d}.csv"
        if not path.exists():
            print(f"Missing: {path}")
            continue
        # auto-detect delimiter
        sample = path.read_text()[:2048]
        delim = ";" if sample.count(";") > sample.count(",") else ","
        df = pd.read_csv(path, sep=delim)
        df["_day"] = d
        df["_global_ts"] = d * 1_000_000 + df["timestamp"].astype(np.int64)
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

trades = load_trades(DATASET_DIR, DAYS)
print(f"Loaded {len(trades):,} trades")
if len(trades):
    print(f"Columns: {list(trades.columns)}")

## 8. Trade volume & notional per product

In [ ]:
if len(trades):
    sym_key = "symbol" if "symbol" in trades.columns else "product"
    t = trades.copy()
    t["quantity"] = pd.to_numeric(t["quantity"], errors="coerce").abs()
    t["price"] = pd.to_numeric(t["price"], errors="coerce")
    t = t.dropna(subset=["quantity", "price"])
    t["notional"] = t["quantity"] * t["price"]
    out = t.groupby(sym_key).agg(volume=("quantity", "sum"), notional=("notional", "sum")).sort_values("volume", ascending=False)
    out["volume"] = out["volume"].astype(int)
    print(out.round(0).to_string())

## 9. Per-family realized vol

1-tick log-return stdev for each variant, scaled to a 10,000-tick day. Helps sort variants by trade-ability vs noise.

In [ ]:
if len(mid_wide):
    log_ret = np.log(mid_wide).diff()
    sd = log_ret.std()
    n = log_ret.count().astype(int)
    per_day = sd * math.sqrt(10_000)
    rep = pd.DataFrame({"n": n, "tick_sd": sd, "per_day_pct": per_day * 100})
    for fam, variants in FAMILIES.items():
        cols = [f"{fam}_{v}" for v in variants if f"{fam}_{v}" in rep.index]
        print(f"\n--- {fam} ---")
        print(rep.loc[cols].round({"tick_sd": 6, "per_day_pct": 2}).to_string())